In [ ]:
%pip install ultralytics opencv-python matplotlib --quiet
%pip install --upgrade pip

In [ ]:
%pip install ipywidgets
%pip install cmake
%pip install dlib

#%pip install face_recognition

In [ ]:
%pip install ipywidgets
%pip install jupyterlab_widgets
%pip install ipyfilechooser


In [ ]:
%pip install opencv-python

# Facial Attendance system

In [ ]:
import cv2
import face_recognition
import pickle
import os
import tkinter as tk
from tkinter import filedialog, simpledialog, messagebox, scrolledtext, Listbox
from ultralytics import YOLO
from datetime import datetime
from PIL import Image, ImageTk
import csv

STUDENT_DIR = "students"
ENCODINGS_FILE = "student_encodings.pkl"
os.makedirs(STUDENT_DIR, exist_ok=True)

class AttendanceApp:
    def __init__(self, root):
        self.root = root
        self.root.title("🎓 Smart Attendance System")
        self.main_menu()

    def main_menu(self):
        self.clear_window()
        tk.Label(self.root, text="📋 Attendance System", font=("Arial", 20)).pack(pady=20)

        tk.Button(self.root, text="➕ Register / Manage Students", width=30, command=self.manage_students).pack(pady=10)
        tk.Button(self.root, text="📁 View Attendance", width=30, command=self.view_attendance).pack(pady=10)
        tk.Button(self.root, text="📷 Start Attendance", width=30, command=self.start_attendance).pack(pady=10)
        tk.Button(self.root, text="❌ Exit", width=30, command=self.root.destroy).pack(pady=10)

    def clear_window(self):
        for widget in self.root.winfo_children():
            widget.destroy()

    # ========== Register / Manage ==========
    def manage_students(self):
        self.clear_window()
        tk.Label(self.root, text="👤 Student Management", font=("Arial", 16)).pack(pady=10)

        listbox = Listbox(self.root, width=50)
        listbox.pack(pady=5)

        try:
            with open(ENCODINGS_FILE, "rb") as f:
                self.encodings = pickle.load(f)
        except FileNotFoundError:
            self.encodings = []

        listbox.delete(0, tk.END)
        for i, student in enumerate(self.encodings):
            name = student.get("name", "Unknown")
            enroll = student.get("enroll", "Unknown")
            listbox.insert(i, f"{enroll} - {name}")

        def add_student():
            file_path = filedialog.askopenfilename(filetypes=[("Image files", "*.jpg *.jpeg *.png")])
            if file_path:
                name = simpledialog.askstring("Name", "Enter Student Name:")
                enroll = simpledialog.askstring("Enrollment Number", "Enter Enrollment Number:")
                if not name or not enroll:
                    messagebox.showwarning("Warning", "Name and Enrollment Number are required!")
                    return

                image = face_recognition.load_image_file(file_path)
                face_enc = face_recognition.face_encodings(image)

                if face_enc:
                    # Remove existing student if re-registering
                    self.encodings = [s for s in self.encodings if s.get("name") != name and s.get("enroll") != enroll]
                    self.encodings.append({"name": name, "enroll": enroll, "encoding": face_enc[0]})
                    with open(ENCODINGS_FILE, "wb") as f:
                        pickle.dump(self.encodings, f)
                    cv2.imwrite(os.path.join(STUDENT_DIR, f"{enroll}_{name}.jpg"), cv2.imread(file_path))
                    messagebox.showinfo("Success", f"{name} ({enroll}) registered/updated!")
                    self.manage_students()
                else:
                    messagebox.showerror("Error", "No face detected in image.")

        def delete_student():
            selected = listbox.curselection()
            if not selected:
                messagebox.showwarning("Warning", "Select a student to delete.")
                return
            line = listbox.get(selected[0])
            enroll = line.split(" - ")[0]
            confirm = messagebox.askyesno("Confirm", f"Delete student with Enrollment No: {enroll}?")
            if confirm:
                self.encodings = [s for s in self.encodings if s.get("enroll") != enroll]
                with open(ENCODINGS_FILE, "wb") as f:
                    pickle.dump(self.encodings, f)
                for file in os.listdir(STUDENT_DIR):
                    if file.startswith(enroll + "_"):
                        os.remove(os.path.join(STUDENT_DIR, file))
                messagebox.showinfo("Deleted", f"Student with Enrollment No: {enroll} removed.")
                self.manage_students()

        tk.Button(self.root, text="➕ Add / Re-register", command=add_student).pack(pady=5)
        tk.Button(self.root, text="🗑️ Delete", command=delete_student).pack(pady=5)
        tk.Button(self.root, text="🔙 Back", command=self.main_menu).pack(pady=10)

    # ========== View Attendance ==========
    def view_attendance(self):
        self.clear_window()
        tk.Label(self.root, text="📁 Attendance Records", font=("Arial", 16)).pack(pady=10)

        text = scrolledtext.ScrolledText(self.root, width=70, height=20)
        text.pack()

        files = [f for f in os.listdir() if f.startswith("attendance_") and f.endswith(".csv")]
        if not files:
            text.insert(tk.END, "No attendance records found.")
        else:
            for f in sorted(files, reverse=True):
                text.insert(tk.END, f"\n=== {f} ===\n")
                with open(f, "r") as file:
                    text.insert(tk.END, file.read() + "\n")

        tk.Button(self.root, text="🔙 Back", command=self.main_menu).pack(pady=10)

    # ========== Start Attendance ==========
    def start_attendance(self):
        self.clear_window()

        try:
            with open(ENCODINGS_FILE, "rb") as f:
                known_faces = pickle.load(f)
        except FileNotFoundError:
            messagebox.showerror("Error", "No students registered.")
            self.main_menu()
            return

        attendance = {}
        model = YOLO("yolov8n-seg.pt")
        cap = cv2.VideoCapture(0)

        if not cap.isOpened():
            messagebox.showerror("Error", "Could not access webcam.")
            return

        def show_frame():
            ret, frame = cap.read()
            if not ret:
                return

            frame = cv2.flip(frame, 1)
            results = model(frame)
            boxes = results[0].boxes.xyxy.cpu().numpy()

            for box in boxes:
                x1, y1, x2, y2 = map(int, box)
                crop = frame[y1:y2, x1:x2]
                rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

                face_locs = face_recognition.face_locations(rgb)
                face_encs = face_recognition.face_encodings(rgb, face_locs)

                recognized = False
                for enc in face_encs:
                    for student in known_faces:
                        if face_recognition.compare_faces([student["encoding"]], enc, tolerance=0.5)[0]:
                            name = student.get("name", "Unknown")
                            enroll = student.get("enroll", "Unknown")
                            if enroll not in attendance:
                                now = datetime.now().strftime("%Y-%m-%d,%H:%M:%S")
                                attendance[enroll] = (name, now)
                            label = f"{name} ({enroll})"
                            color = (0, 255, 0)
                            recognized = True
                            break
                    if recognized:
                        break

                if recognized:
                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
                else:
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
                    cv2.putText(frame, "Not Authorized Person", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            dt = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            cv2.putText(frame, dt, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = ImageTk.PhotoImage(Image.fromarray(img))
            lbl.imgtk = img
            lbl.configure(image=img)
            lbl.after(10, show_frame)

        def save_attendance():
            filename = "attendance_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".csv"
            with open(filename, "w", newline='') as f:
                writer = csv.writer(f)
                writer.writerow(["Enrollment No.", "Name", "Date", "Time"])
                for enroll, (name, datetime_str) in attendance.items():
                    date, time = datetime_str.split(",")
                    writer.writerow([enroll, name, date, time])
            cap.release()
            cv2.destroyAllWindows()
            messagebox.showinfo("Saved", f"Attendance saved to {filename}")
            self.main_menu()

        tk.Label(self.root, text="📷 Live Attendance", font=("Arial", 16)).pack()
        lbl = tk.Label(self.root)
        lbl.pack()

        tk.Button(self.root, text="✅ Finish and Save", command=save_attendance).pack(pady=10)
        tk.Button(self.root, text="🔙 Back", command=lambda: [cap.release(), cv2.destroyAllWindows(), self.main_menu()]).pack()

        show_frame()

if __name__ == "__main__":
    root = tk.Tk()
    app = AttendanceApp(root)
    root.mainloop()
